# 00 — Prepare miniImageNet and persistent DINOv2 features

This notebook is a thin Colab entry point. Reusable code lives in `src/cross_image_glot`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImageGLOT_repo")

if "<YOUR_GITHUB_USERNAME>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImageGLOT_repo
!pip install -q -r requirements.txt

%load_ext autoreload
%autoreload 2

In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

## Purpose

This is the only notebook that touches JPEG images or instantiates DINOv2. It restores the dataset archive from Drive, extracts images locally, restores existing feature shards locally, and computes only missing splits.

In [ ]:
from cross_image_glot.config import EXPECTED_IMAGES
from cross_image_glot.storage import (
    copy_tree_update,
    download_shared_drive_folder_files,
    restore_and_extract_images,
    split_cache_complete,
)

SHARED_FOLDER_ID = "1iDi3zKexa6f3FGTLUVD5SgrnaAa3RlUx"
download_shared_drive_folder_files(SHARED_FOLDER_ID, paths.drive_data_dir)
image_dir = restore_and_extract_images(paths.drive_data_dir, paths.local_data_dir)

# Restore any persistent shards before deciding whether DINOv2 is required.
copy_tree_update(paths.drive_feature_dir, paths.local_feature_dir)
missing_splits = [
    split for split in ("train", "val", "test")
    if not split_cache_complete(paths.local_feature_dir, split, EXPECTED_IMAGES[split])
]
print("Missing feature splits:", missing_splits)

In [ ]:
if missing_splits:
    from cross_image_glot.data import MiniImageNetImageDataset, validate_class_disjointness
    from cross_image_glot.dinov2_cache import (
        DINOv2FeatureCacheBuilder,
        DINOv2FeatureExtractor,
        build_dinov2_transform,
    )

    validate_class_disjointness(paths.local_data_dir)
    transform = build_dinov2_transform(224)
    datasets = {
        split: MiniImageNetImageDataset(paths.local_data_dir, image_dir, split, transform)
        for split in missing_splits
    }
    extractor = DINOv2FeatureExtractor(model_name="dinov2_vits14", image_size=224)
    builder = DINOv2FeatureCacheBuilder(
        extractor=extractor,
        local_cache_dir=paths.local_feature_dir,
        persistent_cache_dir=paths.drive_feature_dir,
        images_per_shard=256,
        extraction_batch_size=32,
        num_workers=2,
    )
    for split in missing_splits:
        builder.build(datasets[split])
else:
    print("All feature splits are complete. DINOv2 was not loaded.")

In [ ]:
for split, expected in EXPECTED_IMAGES.items():
    assert split_cache_complete(paths.local_feature_dir, split, expected)
    assert split_cache_complete(paths.drive_feature_dir, split, expected)
    print(f"{split}: complete locally and in Drive")